# Issue #224 — AlphaGenome predicted-normal validation (Exp 1, patient_001)

**Issue:** [#224](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/224) — *research(filter): patient_001 notebook for AlphaGenome predicted-normal validation (Exp 1+2)*
**Parent:** [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203) — *rethink normal filtering with population panel + AlphaGenome fallback*
**Scope (post 2026-05-16 audit):** Exp 1 only. Exp 2 (germline-aware) carved out to [Sub-Issue #381](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/381) pending WGS acquisition.

## Experiment 1 — Predictive validity (reference-only)

Run AlphaGenome on **GRCh38 reference** (no patient variants) for the patient_001-relevant regions and compare its predicted-junction set against the gold-standard matched-normal RNA-seq.

- **Tissue tracks:** gastric tissue + adjacent stomach (matches patient_001 matched-normal sample)
- **Ground truth:** intersection of matched-normal junctions ∩ GENCODE-annotated (tissue-expressed + annotated only — model not penalised for non-expressed annotation entries)
- **Metric:** precision / recall / F1 across AlphaGenome confidence thresholds
- **PoC scope:** chr22 only (uses [`config/test_config.yaml`](../../config/test_config.yaml) test harness — coord-correct via [PR #372](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/pull/372))
- **Question:** *Does AlphaGenome predict known tissue-expressed splicing from reference alone?*

## Decision rule

Outcome feeds the Exp 1 row of the decision-rule table in [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203). Either outcome (high P/R or low P/R) is publishable per the parent issue's *publishable either way* framing.

## History

- **2026-05-17:** Notebook scaffolded; chr22 test pipeline launched for matched-normal `junctions.tsv` (post-PR #372 coord-correct).
- **2026-05-17:** AlphaGenome SDK env pinned via [PR #386](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/pull/386) ([Sub-Issue #385](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/385)).

In [ ]:
from __future__ import annotations

import gzip
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "workflow").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Coord-correct matched-normal junctions (post-PR #372 bed12_to_junctions.py).
# Produced by: snakemake --cores 4 --use-conda --configfile config/test_config.yaml
MATCHED_NORMAL_TSV = REPO_ROOT / "results" / "patient_001_test" / "alignment" / "SRR9143065_test" / "junctions.tsv"

# chr22-scoped GENCODE v47 basic annotation (built by scripts/prepare_test_data.sh).
GENCODE_GTF = REPO_ROOT / "resources" / "test" / "chr22.gtf.gz"

# AlphaGenome predictions output (TODO: populated by §4 cells).
ALPHAGENOME_OUT_DIR = REPO_ROOT / "results" / "alphagenome" / "issue_224_exp1"
ALPHAGENOME_OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CHROM = "chr22"  # UCSC naming (matches GENCODE GTF + matched-normal BAM)

print(f"Repo root:           {REPO_ROOT}")
print(f"Matched-normal TSV:  {MATCHED_NORMAL_TSV.relative_to(REPO_ROOT)} (exists={MATCHED_NORMAL_TSV.exists()})")
print(f"GENCODE chr22 GTF:   {GENCODE_GTF.relative_to(REPO_ROOT)} (exists={GENCODE_GTF.exists()})")

## §1 — Matched-normal junctions (coord-correct)

Load the post-[PR #372](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/pull/372) `junctions.tsv` produced by [`workflow/scripts/bed12_to_junctions.py`](../../workflow/scripts/bed12_to_junctions.py).

**Format** (2-column TSV, no header): `<chrom>:<donor_1based>:<acceptor_0based_exclusive>:<strand>\t<reads>` — the same format STAR emits via the awk one-liner in [`alignment.smk`](../../workflow/rules/alignment.smk), so the pipeline is aligner-agnostic.

**Coord convention is asymmetric:** donor is 1-based, acceptor is 0-based exclusive (per `bed12_to_junctions.py` line 82; matches `filter_junctions._parse_junction_id`'s `start = int(parts[1]) - 1`, `end = int(parts[2])`). The loader below normalizes to **0-based half-open intron coords** `[donor_0based, acceptor_0based_exclusive)` for set-ops downstream.

**Note:** the raw BED12 (`<sample>_junctions.bed` in the alignment directory) is the **pre-conversion** file — using cols 2-3 directly would re-introduce the [Issue #370](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/370) anchor-outer bug. Always load `junctions.tsv`.

In [ ]:
def load_pipeline_junctions(tsv_path: Path) -> pd.DataFrame:
    """Load the 2-column junctions TSV produced by bed12_to_junctions.py / STAR awk.

    Format: ``<chrom>:<donor_1based>:<acceptor_0based_exclusive>:<strand>\\t<reads>``

    Note the asymmetric coords: donor is 1-based, acceptor is 0-based exclusive
    (per ``workflow/scripts/bed12_to_junctions.py`` line 82; matches
    ``filter_junctions._parse_junction_id``). We normalize to 0-based half-open
    intron ``[donor_0based, acceptor_0based_exclusive)`` for downstream set ops.
    """
    raw = pd.read_csv(tsv_path, sep="\t", header=None, names=["key", "count"])
    parts = raw["key"].str.split(":", expand=True)
    parts.columns = ["chrom", "donor_1based", "acceptor_0based_excl", "strand"]
    out = pd.DataFrame({
        "chrom": parts["chrom"],
        "donor": parts["donor_1based"].astype(int) - 1,           # 0-based intron start
        "acceptor": parts["acceptor_0based_excl"].astype(int),    # 0-based intron end (exclusive)
        "strand": parts["strand"],
        "count": raw["count"].astype(int),
    })
    return out

matched_normal = load_pipeline_junctions(MATCHED_NORMAL_TSV)
print(f"Total matched-normal junctions: {len(matched_normal):,}")
print(f"\nChromosome distribution (sanity — should be {TARGET_CHROM}-only in test config):")
print(matched_normal["chrom"].value_counts().head(10))
print(f"\nRead-count summary (per junction):")
print(matched_normal["count"].describe())
matched_normal.head()

## §2 — GENCODE-annotated junctions for chr22

Parse the GENCODE v47 basic annotation GTF (chr22-filtered by [`scripts/prepare_test_data.sh`](../../scripts/prepare_test_data.sh)) to derive the set of annotated intron junctions.

**Method:** for each transcript, sort its exons by start, then each adjacent exon pair defines an intron `(chrom, exon_i.end, exon_{i+1}.start - 1, strand)`. Donor = first-exon end (0-based intron start); acceptor = second-exon start minus 1 (0-based intron last base).

**Coord convention:** GTF is 1-based inclusive; we convert to 0-based half-open to match the matched-normal TSV. Each intron is `[donor, acceptor)` in 0-based half-open.

In [ ]:
# Parse GTF exons → per-transcript intron set
def parse_exons(gtf_path: Path, target_chrom: str = TARGET_CHROM) -> pd.DataFrame:
    rows = []
    opener = gzip.open if gtf_path.suffix == ".gz" else open
    with opener(gtf_path, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] != "exon" or fields[0] != target_chrom:
                continue
            # GTF is 1-based inclusive
            start_1, end_1 = int(fields[3]), int(fields[4])
            strand = fields[6]
            attrs = fields[8]
            # Extract transcript_id from attribute string
            tx = None
            for kv in attrs.split(";"):
                kv = kv.strip()
                if kv.startswith("transcript_id "):
                    tx = kv.split('"')[1]
                    break
            if tx is None:
                continue
            rows.append({"chrom": fields[0], "start_0": start_1 - 1, "end_0": end_1, "strand": strand, "transcript_id": tx})
    return pd.DataFrame(rows)

exons = parse_exons(GENCODE_GTF)
print(f"chr22 exon records: {len(exons):,}  ({exons['transcript_id'].nunique():,} transcripts)")
exons.head()

In [ ]:
# Derive annotated intron set from sorted exons per transcript.
def exons_to_introns(exons: pd.DataFrame) -> pd.DataFrame:
    introns = []
    for (tx, chrom, strand), grp in exons.groupby(["transcript_id", "chrom", "strand"]):
        sorted_exons = grp.sort_values("start_0").reset_index(drop=True)
        for i in range(len(sorted_exons) - 1):
            donor = int(sorted_exons.loc[i, "end_0"])           # 0-based intron start
            acceptor = int(sorted_exons.loc[i + 1, "start_0"])  # 0-based intron end (exclusive)
            introns.append({"chrom": chrom, "donor": donor, "acceptor": acceptor, "strand": strand})
    df = pd.DataFrame(introns)
    # Dedup across transcripts — same intron in multiple isoforms counts once.
    return df.drop_duplicates().reset_index(drop=True)

annotated = exons_to_introns(exons)
print(f"chr22 annotated introns (unique): {len(annotated):,}")
print(f"Strand split: {annotated['strand'].value_counts().to_dict()}")
annotated.head()

## §3 — Intersect → ground truth (matched-normal ∩ annotated)

Inner-join in `(chrom, donor, acceptor, strand)` tuple space. The intersection is the *tissue-expressed + annotated* set — what AlphaGenome should ideally recover.

**Sanity expectation:** the intersection should contain a meaningful fraction of `matched_normal` (most major isoforms are annotated). Matched-normal junctions *outside* annotated are typically novel splicing or noise — out of scope for the model-not-penalised framing.

In [ ]:
# Build canonical junction tuples for set ops
def junction_keys(df: pd.DataFrame) -> set[tuple]:
    return set(zip(df["chrom"], df["donor"].astype(int), df["acceptor"].astype(int), df["strand"]))

mn_keys = junction_keys(matched_normal)
ann_keys = junction_keys(annotated)
ground_truth_keys = mn_keys & ann_keys

print(f"Matched-normal junctions:           {len(mn_keys):,}")
print(f"Annotated introns (chr22):          {len(ann_keys):,}")
print(f"Ground truth (intersection):        {len(ground_truth_keys):,}")
print(f"Matched-normal covered by annot:    {len(ground_truth_keys) / max(len(mn_keys), 1):.1%}")
print(f"Annotated recovered by matched-norm: {len(ground_truth_keys) / max(len(ann_keys), 1):.1%}")

ground_truth = pd.DataFrame(list(ground_truth_keys), columns=["chrom", "donor", "acceptor", "strand"])
ground_truth.head()

## §4 — AlphaGenome predictions (reference-only)

Run AlphaGenome on the **GRCh38 reference** (no variants) for chr22, scoring junction tracks for **gastric tissue + adjacent stomach** (closest match to patient_001's matched-normal sample).

**Env:** `workflow/envs/alphagenome.yaml` (shipped via [PR #386](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/pull/386)).
**API access:** confirmed in [Sub-Issue #223](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/223) (resolved 2026-05-08).

**Outputs:** per-position splice-site probabilities → filter at threshold τ to produce a junction set, then compare against `ground_truth`.

In [ ]:
# TODO: AlphaGenome client setup.
# Activate the alphagenome env in a separate kernel, or call out via subprocess.
# Example (placeholder — confirm SDK call signature):
#
#     from alphagenome import AlphaGenomeClient
#     client = AlphaGenomeClient(api_key=os.environ["ALPHAGENOME_API_KEY"])
#     tracks = client.list_tissue_tracks(tissue=["stomach", "adjacent_stomach"])
#
# Tissue-track selection: confirm canonical track IDs once the SDK is live.
print("TODO: implement AlphaGenome client init after env is installed locally.")

In [ ]:
# TODO: AlphaGenome chr22 prediction sweep.
# For each chr22 region (split by reasonable window size — confirm SDK limits):
#   1. Submit GRCh38 reference sequence + tissue tracks (stomach + adjacent_stomach)
#   2. Receive per-position splice-donor / splice-acceptor probability tracks
#   3. Threshold and pair donor/acceptor sites within reasonable intron range to derive predicted junctions
#   4. Save raw probabilities + thresholded junction set to ALPHAGENOME_OUT_DIR
#
# Output shape: DataFrame with columns [chrom, donor, acceptor, strand, score]
# matching the matched_normal / annotated frames above for set comparison.
predicted = pd.DataFrame(columns=["chrom", "donor", "acceptor", "strand", "score"])
print(f"Predicted junctions (placeholder): {len(predicted)}")

## §5 — Precision / Recall / F1 across thresholds

Sweep AlphaGenome confidence thresholds τ; for each, compute:

- **TP** = predicted ∩ ground_truth
- **FP** = predicted − ground_truth
- **FN** = ground_truth − predicted
- **Precision** = TP / (TP + FP)
- **Recall** = TP / (TP + FN)
- **F1** = 2·P·R / (P + R)

**Note:** matched-normal junctions *not* in `annotated` are excluded from FN — the model is not penalised for missing non-annotated splicing (per issue scope).

In [ ]:
def pr_f1_at_threshold(predicted: pd.DataFrame, ground_truth_keys: set[tuple], threshold: float) -> dict:
    pred_keys = set(
        zip(
            predicted.loc[predicted["score"] >= threshold, "chrom"],
            predicted.loc[predicted["score"] >= threshold, "donor"].astype(int),
            predicted.loc[predicted["score"] >= threshold, "acceptor"].astype(int),
            predicted.loc[predicted["score"] >= threshold, "strand"],
        )
    )
    tp = len(pred_keys & ground_truth_keys)
    fp = len(pred_keys - ground_truth_keys)
    fn = len(ground_truth_keys - pred_keys)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"threshold": threshold, "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

thresholds = np.linspace(0.0, 1.0, 21)
metrics = pd.DataFrame([pr_f1_at_threshold(predicted, ground_truth_keys, t) for t in thresholds])
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(metrics["recall"], metrics["precision"], marker="o")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall (AlphaGenome chr22 vs matched-normal ∩ annotated)")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

axes[1].plot(metrics["threshold"], metrics["precision"], label="Precision", marker="o")
axes[1].plot(metrics["threshold"], metrics["recall"], label="Recall", marker="s")
axes[1].plot(metrics["threshold"], metrics["f1"], label="F1", marker="^", linewidth=2)
axes[1].set_xlabel("AlphaGenome threshold τ")
axes[1].set_ylabel("Score")
axes[1].set_title("P / R / F1 vs threshold")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## §6 — Decision-rule outcome

Best F1: *(populated after AlphaGenome run)*
Threshold at best F1: *(populated after AlphaGenome run)*
Precision / Recall at best F1: *(populated after AlphaGenome run)*

Feeds the Exp 1 row of the [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203) decision-rule table.

## Ground-truth size on chr22 PoC scope (2026-05-17 smoke-test)

Empirical numbers from `SRR9143065_test` (500K reads, chr22 only):

- Matched-normal junctions: **1,714** (median 1 read, max 23 — sparse coverage)
- GENCODE chr22 annotated unique introns: **7,731**
- Ground truth (matched-normal ∩ annotated): **259** (15.1% of matched-normal, **3.4% of annotated chr22 introns**)

The 3.4% annotated-recovery is a structural floor for the PoC: most chr22 annotated introns belong to transcripts not expressed (or under-detected) in normal stomach at 500K-read depth.

**Implication for metrics — open framing question:**

1. **Restrict evaluable universe to annotated chr22 introns** (7,731): positives = ground truth (259); negatives = annotated minus matched-normal (7,472). AG predicts on the annotated set; P/R measure tissue-expression classification ability. ← *cleanest framing for chr22 PoC*
2. **Whole-genome AG predictions vs. matched-normal ∩ annotated**: any AG prediction outside the 259-set counts as FP, including non-expressed but annotated junctions. Pessimistic; conflates "should predict" with "predicts at all".
3. **Scale up before metrics**: re-run on full FASTQs (production GPU) so the matched-normal sample is denser; only then evaluate metrics on chr22 or genome-wide.

Decision needed before §4-§5 execution. Default scaffold below uses framing (1).

## Cross-references

- [Sub-Issue #381](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/381) — Exp 2 (patient_001 with germline; blocked on WGS)
- [Issue #225](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/225) — Exp 3 (comparative filter strength: matched-normal vs GTEx vs AlphaGenome)
- [Issue #271](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/271) — DISCUSSION manuscript section (deferred until #203 closes)
- [PR #372](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/pull/372) — coord-correct junction extraction (gates the validity of the matched-normal TSV used here)

## Caveats

- **chr22 PoC scope.** Generalisation to full genome is in-scope for the same notebook once the chr22 path is proven; expect different runtime + AlphaGenome cost.
- **Tissue-track approximation.** AlphaGenome's gastric / adjacent_stomach tracks may not perfectly match the patient's normal stomach tissue context — confirm best-available tracks during §4 implementation.
- **Annotated-only ground truth.** Excludes novel splicing in the matched-normal sample; *not* a measure of AlphaGenome's ability to predict novel events.
- **Small ground-truth size at 500K reads.** N=259 positives at chr22 PoC scope; metric variance will be high. Scale-up (full FASTQs on production GPU) likely needed before headline numbers go in the manuscript.